In [ ]:
!pip install transformers datasets librosa jiwer evaluate soundfile torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 45.9 MB/s eta 0:00:00


In [ ]:
import torch
import pandas as pd
import librosa
from datasets import Dataset
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from torch.utils.data import DataLoader



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.12/dist-package

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving audio_path.csv to audio_path.csv
Saving metadata.csv to metadata.csv
Saving transcription.csv to transcription.csv


In [ ]:
df_text = pd.read_csv("transcription.csv")
df_audio = pd.read_csv("audio_path.csv")
df_meta = pd.read_csv("metadata.csv")

# merge on record_id (important)
df = df_audio.merge(df_text, on="record_id")
df=df.head(50)
dataset = Dataset.from_pandas(df[["file_path", "text"]])

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_name = 'openai/whisper-small'
processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name)
model.to(device)


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 768)
      (layers): ModuleList(
        (0-11): 12 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (f

In [1]:
### Mount or Upload audio.zip file to Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
zip_path = "/content/drive/MyDrive/audio.zip"
if os.path.exists(zip_path):
    print("Zip file Found")
else:
    print("Zip file not Found")

Zip file Found


### Unzip file auido.zip

In [ ]:

extract_path="/content/drive/MyDrive/audio"
if os.path.exists(extract_path):
    print("Zip file Found")
else:
    print("Un-Zip the file")
    import zipfile
    with zipfile.ZipFile("/content/drive/MyDrive/audio.zip", 'r') as zip_ref:
        zip_ref.extractall()

Un-Zip the file


In [ ]:
def preprocess(example):
    audio, sr = librosa.load(example['file_path'], sr=16000)
    inputs = processor(audio, sampling_rate=16000)
    example['input_features'] = inputs.input_features[0]
    example['labels'] = processor.tokenizer(example['text']).input_ids
    return example

dataset = dataset.map(preprocess)

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [ ]:
dataset = dataset.train_test_split(test_size=0.1)
train_dataset = dataset['train']
test_dataset = dataset['test']
print(len(dataset))

2


In [ ]:
def collate_fn(batch):
    input_features = [torch.tensor(x['input_features']) for x in batch]
    input_features = torch.nn.utils.rnn.pad_sequence(input_features, batch_first=True)

    labels = [torch.tensor(x['labels']) for x in batch]
    labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)

    return {'input_features': input_features, 'labels': labels}

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
model.train()
for batch in train_loader:
    input_features = batch['input_features'].to(device)
    labels = batch['labels'].to(device)

    outputs = model(input_features=input_features, labels=labels)
    loss = outputs.loss

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    print('Loss:', loss.item())


Loss: 2.955101490020752
Loss: 3.2911009788513184
Loss: 2.5700252056121826
Loss: 2.879936695098877
Loss: 4.400132656097412
Loss: 2.8401598930358887
Loss: 3.5016040802001953
Loss: 2.1054768562316895
Loss: 2.503016710281372
Loss: 2.4383859634399414
Loss: 3.4566256999969482
Loss: 2.6004629135131836
Loss: 2.461726665496826
Loss: 2.6051206588745117
Loss: 2.1661226749420166
Loss: 4.351574897766113
Loss: 5.12672233581543
Loss: 3.258540153503418
Loss: 2.0538384914398193
Loss: 2.498363494873047
Loss: 2.187588930130005
Loss: 2.3234477043151855
Loss: 4.357511520385742
Loss: 2.245835065841675
Loss: 2.8801140785217285
Loss: 2.056018114089966
Loss: 2.4304347038269043
Loss: 2.219816207885742
Loss: 2.3983328342437744
Loss: 1.9435710906982422
Loss: 2.186199903488159
Loss: 2.0688390731811523
Loss: 2.1009700298309326
Loss: 2.1395115852355957
Loss: 2.30430269241333
Loss: 2.044329881668091
Loss: 1.8578579425811768
Loss: 2.2958879470825195
Loss: 2.6728620529174805
Loss: 2.160524368286133
Loss: 2.199826478958

In [ ]:
model.save_pretrained('model_final')
processor.save_pretrained('model_final')


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['model_final/processor_config.json']

In [ ]:
def transcribe_file(model,processor,file_path):
    import librosa
    import torch
    audio, sr = librosa.load(file_path, sr=16000)

    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    )

    input_features = inputs.input_features.to(model.device)
    model.eval()
    with torch.no_grad():
        predicted_ids = model.generate(input_features)

    return processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

In [ ]:
predictions = []
references = []
test=df
for i in range(5):
    file_path = df.iloc[i]["file_path"]
    true_text = df.iloc[i]["text"]

    pred = transcribe_file(model,processor,file_path)

    predictions.append(pred)
    references.append(true_text)

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.log

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    return text

predictions = [clean_text(p) for p in predictions]
references = [clean_text(r) for r in references]

In [ ]:
from evaluate import load

wer_metric = load("wer")

wer = wer_metric.compute(
    predictions=predictions,
    references=references
)

print("WER:", wer)

WER: 2.814814814814815


Model Save and Download

In [ ]:
import shutil

shutil.make_archive("model_final", 'zip', "model_final")

print("✅ model_final.zip created")

In [ ]:
from google.colab import files
files.download('model_final.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>